In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('feature_combination_summary.csv')

metrics_to_avg = ['le2_pct', 'le5_pct', 'le10_pct']
df_agg = df.groupby(['snr_db', 'feature_set'])[metrics_to_avg].mean().reset_index()

def categorize_features(feat):
    n_plus = str(feat).count('+')
    if n_plus == 0:
        return 'Single Features'
    elif n_plus == 1:
        return 'Double Features'
    else:
        return 'Triple & Quad Features'

df_agg['category'] = df_agg['feature_set'].apply(categorize_features)

def plot_category_metrics(category_name, df_cat):
    features = sorted(df_cat['feature_set'].unique())

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Probability Metrics for {category_name}', fontsize=16, fontweight='bold', y=1.05)

    metrics_info = [
        ('le2_pct', 'Probability ≤ 2ms (%)', 'Blues'),
        ('le5_pct', 'Probability ≤ 5ms (%)', 'Greens'),
        ('le10_pct', 'Probability ≤ 10ms (%)', 'Purples')
    ]

    palette_name = 'tab20' if len(features) > 10 else 'tab10'
    snr_ticks = sorted(df['snr_db'].unique())

    for idx, (metric, ylabel, cmap_hint) in enumerate(metrics_info):
        ax = axes[idx]

        sns.lineplot(
            data=df_cat,
            x='snr_db',
            y=metric,
            hue='feature_set',
            ax=ax,
            palette=palette_name,
            linewidth=1.5,
            alpha=0.85,
            markers=True,
            markersize=5
        )

        ax.set_title(ylabel, fontsize=12, fontweight='bold')
        ax.set_xlabel('SNR (dB)', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)

        ax.set_xticks(snr_ticks)
        ax.tick_params(axis='x', rotation=45)

        if idx == 2:
            ax.legend(title='Feature Combination', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9, title_fontsize=10)
        else:
            ax.get_legend().remove()

        ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout(rect=[0, 0, 0.90, 0.95])

    safe_name = category_name.replace(' ', '_').replace('&', 'and')
    plt.savefig(f'{safe_name}_probability_metrics.png', dpi=300, bbox_inches='tight')
    print(f"Saved: {safe_name}_probability_metrics.png")
    plt.show()

for cat in ['Single Features', 'Double Features', 'Triple & Quad Features']:
    df_cat = df_agg[df_agg['category'] == cat]
    if not df_cat.empty:
        print(f"\nPlotting {cat}... (Total combinations: {len(df_cat['feature_set'].unique())})")
        plot_category_metrics(cat, df_cat)
